# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Neel0289/FlyRank-Week1A1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — Growth Prediction

The paper reports that its growth model reached about 90% accuracy on unseen pages from the same brands and about 75% on brands that were not seen during training. The paper also reports ranges across repeated tests.

**Methodology question:** How exactly was the "growing vs declining" label defined, including the future outcome window used to create it? I would want to verify that the label is based only on information after the feature window so that the predictors cannot contain future information.

**Validation question:** The paper reports testing on unseen pages and unseen brands. I would want to confirm that the grouped validation prevents pages from the same brand from appearing in both training and test sets, and that the reported range across repeated tests supports the strength of the headline accuracy claim.

### Finding 2 — Refreshing Pages Actually Works

The paper reports that 7 of 9 tested strata showed statistically significant refresh lift and reports large median differences between refreshed and stale pages in several age/competition segments.

**Methodology question:** How were "refreshed" and "stale" pages operationally defined, and how was selection bias handled? Pages selected for refresh may already differ from pages that were not refreshed in traffic, age, strategic value, or prior performance.

**Validation question:** Does the held-out comparison establish a causal effect of refreshing, or should the result be interpreted more narrowly as an observed association between refresh status and later performance?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

My Week-5 model used a grouped client split, which was a good validation choice, but its proxy relevance score was constructed directly from engagement_rate, scroll_rate, and trend_pct while those same fields were also supplied to the Random Forest. This created target-feature leakage and likely inflated the NDCG result of 0.9989.

For the audit, I keep the client-grouped split but separate the proxy-target inputs from the model feature set. I also construct the proxy independently within the train and test partitions. The comparison below shows the Week-5 result before the audit and the corrected result after the leakage fix.

The corrected result should be interpreted as directional evidence about this proxy-ranking setup, not as proof of real future opportunity prediction.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import ndcg_score

# --------------------------------------------------
# Load data
# --------------------------------------------------

df = pd.read_csv(
    "../../data/raw/content_refresh_anonymized.csv"
)

# --------------------------------------------------
# Target-only variables
# These are NOT supplied to the model.
# --------------------------------------------------

target_cols = [
    "engagement_rate",
    "scroll_rate",
    "trend_pct"
]

# --------------------------------------------------
# Model feature set
# Explicitly excludes target components.
# --------------------------------------------------

feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "search_volume",
    "ctr",
    "avg_position",
    "ai_traffic_pct",
    "content_age_days",
    "days_since_last_update"
]

required_cols = (
    feature_cols
    + target_cols
    + ["client_id"]
)

model_df = df.dropna(
    subset=required_cols
).copy()

# --------------------------------------------------
# Honest grouped split
# --------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        groups=model_df["client_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

# --------------------------------------------------
# Build proxy separately within each split
# --------------------------------------------------

def build_proxy(data):
    return (
        data["engagement_rate"].rank(pct=True) * 0.5
        + data["scroll_rate"].rank(pct=True) * 0.2
        + data["trend_pct"].rank(pct=True) * 0.3
    )

train_df["proxy_relevance"] = build_proxy(train_df)
test_df["proxy_relevance"] = build_proxy(test_df)

# --------------------------------------------------
# Confirm no client overlap
# --------------------------------------------------

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print(
    "Client overlap:",
    len(train_clients & test_clients)
)

# --------------------------------------------------
# Train corrected model
# --------------------------------------------------

rf = RandomForestRegressor(
    n_estimators=25,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    train_df[feature_cols],
    train_df["proxy_relevance"]
)

test_df["model_score"] = rf.predict(
    test_df[feature_cols]
)

# --------------------------------------------------
# NDCG
# --------------------------------------------------

corrected_ndcg = ndcg_score(
    [test_df["proxy_relevance"].to_numpy()],
    [test_df["model_score"].to_numpy()],
    k=20
)

before_ndcg = 0.998915

comparison = pd.DataFrame({
    "Evaluation": [
        "Week-5 result before audit",
        "Week-6 corrected result"
    ],
    "NDCG@20": [
        before_ndcg,
        corrected_ndcg
    ]
})

display(comparison)

print(
    f"Change after leakage correction: "
    f"{corrected_ndcg - before_ndcg:.4f}"
)

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --------------------------------------------------
# Leakage audit
# --------------------------------------------------

leakage_keywords = [
    "label",
    "target",
    "future",
    "outcome",
    "opportunity"
]

print("Final model features:")
for col in feature_cols:
    print("-", col)

print("\nTarget construction fields:")
for col in target_cols:
    print("-", col)

overlap = set(feature_cols) & set(target_cols)

print("\nTarget-feature overlap:")
print(overlap)

if overlap:
    print("FAIL: target components are still in model features.")
else:
    print("PASS: target construction fields are excluded from model features.")

# Check forbidden field names
flagged = [
    col for col in feature_cols
    if any(
        term in col.lower()
        for term in leakage_keywords
    )
]

print("\nPotentially suspicious feature names:")
print(flagged if flagged else "None found")

## 3. Leakage audit

The Week-5 audit identified a genuine target-feature overlap: the proxy relevance score was constructed from engagement_rate, scroll_rate, and trend_pct, while those same fields were supplied to the model. I therefore do not treat the original NDCG@20 result as trustworthy evidence of model quality.

After correction, the proxy-target fields are excluded from the model feature set and the proxy is constructed separately within the train and test partitions. The final check confirms that no target-construction fields remain in the model inputs.

This does not prove that the final feature set is completely free of every possible form of leakage; it shows that the specific leakage pathway identified in Week 5 has been removed.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original claim

"The Random Forest improves ranking quality over the Week-4 baseline."

### Revised claim

"Under the grouped client split, the corrected Random Forest produced a measured NDCG@20 score that can be compared directionally with the Week-4 baseline. Because the relevance target is a constructed proxy rather than an observed future outcome, the result is best treated as decision-support evidence rather than proof of improved real-world opportunity prediction."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.